# Similarity-Pruned Dataset Workflow

Select one random anchor graph, compute vectorized embeddings for all graphs,
define a regression target as cosine similarity to that anchor,
remove the top-`k` most similar graphs, and continue with the pruned dataset.


In [ ]:
# Import shared helper modules and set up the repository path once at the top.
import sys
from pathlib import Path

for _root in [Path.cwd(), *Path.cwd().parents[:2]]:
    if (_root / 'conditional_node_field_graph_generator').exists():
        _root_str = str(_root.resolve())
        if _root_str not in sys.path:
            sys.path.insert(0, _root_str)
        break
else:
    raise ModuleNotFoundError("Could not locate 'conditional_node_field_graph_generator' package from current working directory.")

del _root, _root_str

from sklearn.metrics.pairwise import cosine_similarity
from NSPPK.nsppk import NSPPK

from conditional_node_field_graph_generator.extensions.demo.pipeline import (
    build_dataset,
    build_graph_generator,
    fit_graph_generator,
)
from conditional_node_field_graph_generator.extensions.demo.storage import (
    find_latest_checkpoint,
    list_training_checkpoints,
)
from conditional_node_field_graph_generator.extensions.demo.visualization import (
    infer_display_mode,
    plot_networkx_graphs,
    plot_similarity_distribution_with_iqr,
)
from conditional_node_field_graph_generator.extensions.molecular import draw_molecules
from conditional_node_field_graph_generator.persistence import (
    list_saved_graph_generators,
    load_graph_generator,
    save_graph_generator,
)


# Auxiliary code

## Environment And Utility Setup
Load the shared helpers and utility functions used throughout the similarity-pruned workflow.


In [ ]:
# Configuration moved below the experiment title.


In [ ]:
# Compute graph embeddings used for similarity pruning and evaluation.
def compute_graph_embeddings(graphs, nbits=11):
    graph_vectorizer = NSPPK(
        radius=2,
        distance=4,
        connector=1,
        nbits=nbits,
        dense=True,
        parallel=True,
        use_edges_as_features=True,
    )
    embeddings = np.asarray(graph_vectorizer.fit_transform(graphs), dtype=np.float32)
    return graph_vectorizer, embeddings


def prune_by_anchor_similarity(graphs, targets, embeddings, k_remove=5, random_state=42, remove_anchor=False):
    if len(graphs) == 0:
        raise ValueError("Cannot prune an empty dataset.")
    if k_remove < 0:
        raise ValueError("k_remove must be >= 0.")
    if k_remove >= len(graphs):
        raise ValueError("k_remove must be smaller than the dataset size.")

    rng = np.random.default_rng(random_state)
    anchor_idx = int(rng.integers(low=0, high=len(graphs)))

    anchor_vec = embeddings[anchor_idx:anchor_idx + 1]
    similarity = cosine_similarity(embeddings, anchor_vec).ravel()

    ranking = np.argsort(-similarity)
    ordered = [idx for idx in ranking if (remove_anchor or idx != anchor_idx)]
    remove_indices = np.asarray(ordered[:k_remove], dtype=int)

    keep_mask = np.ones(len(graphs), dtype=bool)
    keep_mask[remove_indices] = False

    pruned_graphs = [g for g, keep in zip(graphs, keep_mask) if keep]
    pruned_targets = np.asarray([t for t, keep in zip(targets, keep_mask) if keep])
    pruned_similarity = similarity[keep_mask]

    removed_similarity = similarity[remove_indices]

    summary = pd.DataFrame({
        'graph_index': np.arange(len(graphs)),
        'target': targets,
        'cosine_to_anchor': similarity,
        'is_anchor': np.arange(len(graphs)) == anchor_idx,
        'removed': ~keep_mask,
    }).sort_values('cosine_to_anchor', ascending=False)

    return {
        'anchor_idx': anchor_idx,
        'anchor_graph': graphs[anchor_idx],
        'anchor_target': targets[anchor_idx],
        'similarity_target': similarity,
        'remove_indices': remove_indices,
        'removed_similarity': removed_similarity,
        'summary': summary,
        'pruned_graphs': pruned_graphs,
        'pruned_targets': pruned_targets,
        'pruned_similarity_target': pruned_similarity,
        'keep_mask': keep_mask,
    }


## Strict CFG Experiment: Similarity to Hidden Target Graph

Train `ConditionalNodeFieldGraphGenerator` on the pruned dataset with the regression target
`working_similarity_target`, then generate with `desired_target=1.0` and `desired_target=0.7`
and compare average cosine similarity to the hidden anchor graph.


### Configuration
Set artifact locations and the directory used to save fitted generators.


In [ ]:
# Configuration

REPO_ROOT = next(
    candidate.resolve()
    for candidate in [Path.cwd(), Path.cwd().parent]
    if (candidate / 'conditional_node_field_graph_generator').exists()
)
ARTIFACT_ROOT = REPO_ROOT / '.artifacts'
SAVED_GENERATOR_ROOT = ARTIFACT_ROOT / 'saved_generators'


In [ ]:
# Import shared generator persistence/checkpoint helpers and define the similarity-pruned dataset builder.
    build_graph_generator,
    fit_graph_generator,
)
    find_latest_checkpoint,
    list_training_checkpoints,
)
    list_saved_graph_generators,
    load_graph_generator,
    save_graph_generator,
)

def build_similarity_optimization_dataset(
    dataset_type='MOLECULAR',
    dataset_size=2000,
    size=30,
    k_remove=5,
    assay_id='651610',
    embedding_nbits=11,
    random_state=42,
    remove_anchor=False,
):
    graphs, targets = build_dataset(
        dataset_type=dataset_type,
        dataset_size=dataset_size,
        size=size,
        assay_id=assay_id,
    )
    print(f"Loaded graphs: {len(graphs)}")
    if len(targets) > 0:
        target_array = np.asarray(targets)
        if np.issubdtype(target_array.dtype, np.integer):
            print(f"Class split: {np.bincount(target_array.astype(int))}")
    else:
        target_array = np.asarray(targets)

    graph_vectorizer, embeddings = compute_graph_embeddings(graphs, nbits=embedding_nbits)
    print(f"Embeddings shape: {embeddings.shape}")

    result = prune_by_anchor_similarity(
        graphs=graphs,
        targets=target_array,
        embeddings=embeddings,
        k_remove=k_remove,
        random_state=random_state,
        remove_anchor=remove_anchor,
    )

    display(result['summary'].head(12))
    display_mode = infer_display_mode(graphs)
    print("Anchor graph:")
    plot_networkx_graphs([result['anchor_graph']], n_cols=1, mode=display_mode)

    removed_graphs = [graphs[i] for i in result['remove_indices']]
    print(f"Removed top-{len(removed_graphs)} most similar graphs:")
    plot_networkx_graphs(removed_graphs, n_cols=max(1, len(removed_graphs)), mode=display_mode)

    working_graphs = result['pruned_graphs']
    working_targets = result['pruned_targets']
    working_similarity_target = result['pruned_similarity_target']
    print(f"working_graphs: {len(working_graphs)}")
    print(f"working_targets: {len(working_targets)}")
    print(f"working_similarity_target shape: {working_similarity_target.shape}")
    print(f"cosine target range: [{working_similarity_target.min():.4f}, {working_similarity_target.max():.4f}]")

    return working_graphs, working_similarity_target, result['anchor_graph'], graph_vectorizer


def cosine_to_reference_graph(graphs_to_compare, reference_graph, fitted_graph_vectorizer):
    if len(graphs_to_compare) == 0:
        return np.array([], dtype=float)
    reference_embedding = np.asarray(fitted_graph_vectorizer.transform([reference_graph]), dtype=np.float32)
    generated_embeddings = np.asarray(fitted_graph_vectorizer.transform(graphs_to_compare), dtype=np.float32)
    return cosine_similarity(generated_embeddings, reference_embedding).ravel()


def cosine_to_hidden_target(graphs_to_compare, hidden_graph, fitted_graph_vectorizer):
    return cosine_to_reference_graph(graphs_to_compare, hidden_graph, fitted_graph_vectorizer)


---

# Experiment

### Build The Pruned Dataset
Create the anchor-based similarity-pruned dataset that the model will train on.


In [ ]:
# Construct the pruned similarity dataset and expose the working graphs and targets.
OPTIMIZATION_DATASET_TYPE = 'MOLECULAR'
OPTIMIZATION_DATASET_SIZE = 6000
OPTIMIZATION_SIZE = 25
OPTIMIZATION_K_REMOVE = 5

working_graphs, working_similarity_target, anchor_graph, graph_vectorizer = build_similarity_optimization_dataset(
    dataset_type=OPTIMIZATION_DATASET_TYPE,
    dataset_size=OPTIMIZATION_DATASET_SIZE,
    size=OPTIMIZATION_SIZE,
    k_remove=OPTIMIZATION_K_REMOVE,
)

### Train And Save The Generator
Fit the generator on the pruned dataset, then save it with a short reusable filename.


In [ ]:
# Build, fit, and save the similarity-conditioned graph generator.
%%time
MODEL_NAME = f"demo-optimization-{OPTIMIZATION_DATASET_TYPE.lower()}-n{len(working_graphs)}-size{OPTIMIZATION_SIZE}"
similarity_graph_generator = build_graph_generator(
    # General
    verbose=2,
    nbits=11,
    # Network architecture
    latent_embedding_dimension=96,
    number_of_transformer_layers=3,
    transformer_attention_head_count=4,
    transformer_dropout=0.15,
    # Training
    learning_rate=1e-4,
    maximum_epochs=256,
    batch_size=16,
    total_steps=100,
    verbose_epoch_interval=10,
    enable_early_stopping=True,
    early_stopping_monitor="val_total",
    early_stopping_mode="min",
    early_stopping_patience=20,
    early_stopping_min_delta=5.2e4 * 1e-3,
    early_stopping_ema_alpha=0.3,
    restore_best_checkpoint=True,
    important_feature_index=1,
    # Loss weights
    lambda_degree_importance=2.0,
    default_exist_pos_weight=1.0,
    lambda_node_exist_importance=2.0,
    lambda_node_count_importance=0.5,
    lambda_node_label_importance=2.0,
    lambda_edge_label_importance=2.0,
    lambda_direct_edge_importance=2.0,
    lambda_edge_count_importance=0.5,
    lambda_degree_edge_consistency_importance=0.5,
    lambda_auxiliary_edge_importance=1.0,
    # Sampling and guidance
    degree_temperature=1,
    pool_condition_tokens=False,
    node_field_sigma=0.2,
    sampling_step_size=0.05,
    sampling_steps=None,
    langevin_noise_scale=0.0,
    cfg_condition_dropout_prob=0.1,
    cfg_null_target_strategy="zero",
    # Locality supervision and generation
    locality_horizon=1,
    locality_sample_fraction=0.5,
    negative_sample_factor=1,
    locality_sampling_strategy="stratified_preserve",
    locality_target_positive_ratio=0.5,
    use_feasibility_filtering=True,
    max_feasibility_attempts=10,
    feasibility_candidates_per_attempt=4,
    feasibility_failure_mode="return_partial",
    # Decoder
    decoder_existence_threshold=0.5,
    decoder_enforce_connectivity=True,
    decoder_degree_slack_penalty=1e6,
    decoder_warm_start_mst=True,
    decoder_n_jobs=1,
    # Outputs
    artifact_root=REPO_ROOT / ".artifacts",
    checkpoint_root=REPO_ROOT / ".artifacts" / "checkpoints" / "node_field",
    model_name=MODEL_NAME,
    model_dir=SAVED_GENERATOR_ROOT,
)

print(f"Training on {len(working_graphs)} pruned graphs with similarity targets in [{working_similarity_target.min():.4f}, {working_similarity_target.max():.4f}]")
similarity_graph_generator = fit_graph_generator(
    similarity_graph_generator,
    working_graphs,
    targets=working_similarity_target,
    resume_latest_checkpoint=True,
    checkpoint_root=REPO_ROOT / ".artifacts" / "checkpoints" / "node_field",
)

# Save trained generator

MODEL_FILENAME = save_graph_generator(similarity_graph_generator)
MODEL_FILENAME


### Sample From The Saved Or Current Generator
Resume from a saved filename if needed, then generate samples at different similarity targets.


In [ ]:
# Resume later with a copied filename from the save cell, or inspect Lightning checkpoints.
# list_saved_graph_generators(SAVED_GENERATOR_ROOT)
# list_training_checkpoints(REPO_ROOT / ".artifacts" / "checkpoints" / "node_field")
# print(find_latest_checkpoint(REPO_ROOT / ".artifacts" / "checkpoints" / "node_field"))
# similarity_graph_generator = load_graph_generator(MODEL_FILENAME, model_dir=SAVED_GENERATOR_ROOT)


apply_feasibility_filtering = False
n_samples = 14

generated_high = similarity_graph_generator.sample_conditioned_on_random(
    working_graphs,
    n_samples=n_samples,
    desired_target=1.0,
    guidance_scale=4.0,
    apply_feasibility_filtering=apply_feasibility_filtering,
)

generated_low = similarity_graph_generator.sample_conditioned_on_random(
    working_graphs,
    n_samples=n_samples,
    desired_target=0.55,
    guidance_scale=4.0,
    apply_feasibility_filtering=apply_feasibility_filtering,
)

# Similarity vectors for generated samples vs hidden target
sim_high = cosine_to_hidden_target(generated_high, anchor_graph, graph_vectorizer)
sim_low = cosine_to_hidden_target(generated_low, anchor_graph, graph_vectorizer)

# Plot + summary stats in one utility call
similarity_stats = plot_similarity_distribution_with_iqr(
    sim_high=sim_high,
    sim_low=sim_low,
    target_high=1.0,
    target_low=0.55,
)


### Visual Comparison
Render the sampled graphs and inspect how the similarity target changes the decoded outputs.


In [ ]:
# Visualize the generated samples and compare them against the hidden target graph.
display_mode = infer_display_mode(working_graphs)

high_order = np.argsort(-sim_high)
low_order = np.argsort(-sim_low)
sorted_generated_high = [generated_high[index] for index in high_order]
sorted_generated_low = [generated_low[index] for index in low_order]
sorted_sim_high = sim_high[high_order]
sorted_sim_low = sim_low[low_order]

print("Anchor graph:")
plot_networkx_graphs([anchor_graph], n_cols=1, mode=display_mode)

print('Sample generations for desired_target=1.0')
print(sorted_sim_high)
plot_networkx_graphs(
    sorted_generated_high,
    n_cols=min(6, max(1, len(sorted_generated_high[:6]))),
    mode=display_mode,
    titles=[f"sim={value:.3f}" for value in sorted_sim_high[:6]],
)

print('Sample generations for desired_target=0.7')
print(sorted_sim_low)
plot_networkx_graphs(
    sorted_generated_low,
    n_cols=min(6, max(1, len(sorted_generated_low[:6]))),
    mode=display_mode,
    titles=[f"sim={value:.3f}" for value in sorted_sim_low[:6]],
)


### Separate Classifier Guidance Experiment
Pick a fresh random probe molecule, define a binary task from cosine similarity to that probe, train the post-hoc guidance classifier, and compare high-similarity vs low-similarity generations.


In [ ]:
# Build a new binary optimization task from similarity to a fresh probe molecule.
classifier_probe_rng = np.random.default_rng(123)
classifier_probe_index = int(classifier_probe_rng.integers(low=0, high=len(working_graphs)))
classifier_probe_graph = working_graphs[classifier_probe_index]
classifier_probe_similarity = cosine_to_reference_graph(working_graphs, classifier_probe_graph, graph_vectorizer)
classifier_probe_threshold = float(np.median(classifier_probe_similarity))
classifier_guidance_targets = (classifier_probe_similarity >= classifier_probe_threshold).astype(int)

print(f'Probe index: {classifier_probe_index}')
print(f'Binary similarity threshold: {classifier_probe_threshold:.4f}')
print('Class split [low, high]:', np.bincount(classifier_guidance_targets, minlength=2))
print(f'Similarity range to probe: [{classifier_probe_similarity.min():.4f}, {classifier_probe_similarity.max():.4f}]')

display_mode = infer_display_mode(working_graphs)
print('Probe graph:')
plot_networkx_graphs([classifier_probe_graph], n_cols=1, mode=display_mode)

pd.Series(classifier_probe_similarity).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])


In [ ]:
%%time
# Train the post-hoc classifier-guidance head for the new binary similarity task.
similarity_graph_generator.set_guidance_classifier(num_classes=2)
similarity_graph_generator.train_guidance_classifier(
    working_graphs,
    targets=classifier_guidance_targets,
    learning_rate=1e-3,
    maximum_epochs=40,
    batch_size=32,
    noise_scale=0.15,
)


In [ ]:
# Generate molecules guided toward high vs low similarity to the fresh probe molecule.
classifier_guidance_scale = 3.0
classifier_generated_high = similarity_graph_generator.sample_conditioned_on_random_classifier_guided(
    working_graphs,
    desired_class=1,
    n_samples=n_samples,
    classifier_scale=classifier_guidance_scale,
    apply_feasibility_filtering=apply_feasibility_filtering,
)

classifier_generated_low = similarity_graph_generator.sample_conditioned_on_random_classifier_guided(
    working_graphs,
    desired_class=0,
    n_samples=n_samples,
    classifier_scale=classifier_guidance_scale,
    apply_feasibility_filtering=apply_feasibility_filtering,
)

classifier_sim_high = cosine_to_reference_graph(classifier_generated_high, classifier_probe_graph, graph_vectorizer)
classifier_sim_low = cosine_to_reference_graph(classifier_generated_low, classifier_probe_graph, graph_vectorizer)

classifier_similarity_stats = plot_similarity_distribution_with_iqr(
    sim_high=classifier_sim_high,
    sim_low=classifier_sim_low,
    target_high='classifier-high',
    target_low='classifier-low',
)
classifier_similarity_stats


In [ ]:
# Visualize classifier-guided generations sorted by similarity to the fresh probe molecule.
classifier_high_order = np.argsort(-classifier_sim_high)
classifier_low_order = np.argsort(-classifier_sim_low)
classifier_sorted_generated_high = [classifier_generated_high[index] for index in classifier_high_order]
classifier_sorted_generated_low = [classifier_generated_low[index] for index in classifier_low_order]
classifier_sorted_sim_high = classifier_sim_high[classifier_high_order]
classifier_sorted_sim_low = classifier_sim_low[classifier_low_order]

print('Probe graph:')
plot_networkx_graphs([classifier_probe_graph], n_cols=1, mode=display_mode)

print('Classifier-guided generations toward high similarity')
print(classifier_sorted_sim_high)
plot_networkx_graphs(
    classifier_sorted_generated_high,
    n_cols=min(6, max(1, len(classifier_sorted_generated_high[:6]))),
    mode=display_mode,
    titles=[f'sim={value:.3f}' for value in classifier_sorted_sim_high[:6]],
)

print('Classifier-guided generations toward low similarity')
print(classifier_sorted_sim_low)
plot_networkx_graphs(
    classifier_sorted_generated_low,
    n_cols=min(6, max(1, len(classifier_sorted_generated_low[:6]))),
    mode=display_mode,
    titles=[f'sim={value:.3f}' for value in classifier_sorted_sim_low[:6]],
)
